In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import defaultdict
import random
import numpy as np


In [2]:
class SkipGram(nn.Module):
  def __init__(self, vocab_size, embed_dim):
    super(SkipGram, self).__init__()
    self.embeddings = nn.Embedding(vocab_size, embed_dim)
    self.output = nn.Linear(embed_dim, vocab_size)
  def forward(self, input):
    embed = self.embeddings(input)
    out = self.output(embed)
    return out



In [3]:
def train(model, data, epochs=100, lr=0.01):
  optimizer = optim.SGD(model.parameters(), lr=lr)
  loss_fn = nn.CrossEntropyLoss()

  for epoch in range(epochs):
    total_loss = 0
    for target, context in data:
      model.zero_grad()
      output = model(torch.tensor([target]))
      loss = loss_fn(output, torch.tensor([context]))
      loss.backward()
      optimizer.step()
      total_loss += loss.item()
    print(f'Epoch {epoch}, Loss: {total_loss:.4f}')

In [4]:
corpus = [
    "I love to watch movies in the evening",
    "I love to read books in the morning",
    "movies and books are great for relaxation",
    "reading is a great way to learn new things"
]

# 1. Tokenize text and build vocabulary
word_to_idx = {}
idx_to_word = {}
vocab = set()

for sentence in corpus:
    for word in sentence.lower().split():
        vocab.add(word)

for i, word in enumerate(sorted(list(vocab))):
    word_to_idx[word] = i
    idx_to_word[i] = word

vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")
print(f"Word to index mapping: {word_to_idx}")


Vocabulary size: 23
Word to index mapping: {'a': 0, 'and': 1, 'are': 2, 'books': 3, 'evening': 4, 'for': 5, 'great': 6, 'i': 7, 'in': 8, 'is': 9, 'learn': 10, 'love': 11, 'morning': 12, 'movies': 13, 'new': 14, 'read': 15, 'reading': 16, 'relaxation': 17, 'the': 18, 'things': 19, 'to': 20, 'watch': 21, 'way': 22}


In [5]:
def generate_pairs(corpus, word_to_idx, window_size=2):
    data = []
    for sentence in corpus:
        tokens = sentence.lower().split()
        indices = [word_to_idx[word] for word in tokens]
        for i, target_idx in enumerate(indices):
            for j in range(max(0, i - window_size), min(len(indices), i + window_size + 1)):
                if i != j:
                    context_idx = indices[j]
                    data.append((target_idx, context_idx))
    return data


training_data = generate_pairs(corpus, word_to_idx, window_size=2)
print(f"Number of training pairs: {len(training_data)}")
print(f"Sample training pairs: {training_data[:5]}")


Number of training pairs: 104
Sample training pairs: [(7, 11), (7, 20), (11, 7), (11, 20), (11, 21)]


In [6]:
embed_dim = 10
model = SkipGram(vocab_size, embed_dim)

print("Starting training...")
train(model, training_data, epochs=500, lr=0.01)
print("Training complete!")

# Get word embeddings
word_embeddings = model.embeddings.weight.detach().numpy()
print("\nWord Embeddings:")
for word, idx in word_to_idx.items():
    print(f"{word}: {word_embeddings[idx]}")


Starting training...
Epoch 0, Loss: 342.9592
Epoch 1, Loss: 331.7298
Epoch 2, Loss: 322.0290
Epoch 3, Loss: 313.6042
Epoch 4, Loss: 306.2570
Epoch 5, Loss: 299.8238
Epoch 6, Loss: 294.1657
Epoch 7, Loss: 289.1633
Epoch 8, Loss: 284.7142
Epoch 9, Loss: 280.7311
Epoch 10, Loss: 277.1413
Epoch 11, Loss: 273.8840
Epoch 12, Loss: 270.9094
Epoch 13, Loss: 268.1763
Epoch 14, Loss: 265.6508
Epoch 15, Loss: 263.3047
Epoch 16, Loss: 261.1142
Epoch 17, Loss: 259.0597
Epoch 18, Loss: 257.1243
Epoch 19, Loss: 255.2936
Epoch 20, Loss: 253.5555
Epoch 21, Loss: 251.8994
Epoch 22, Loss: 250.3163
Epoch 23, Loss: 248.7982
Epoch 24, Loss: 247.3385
Epoch 25, Loss: 245.9311
Epoch 26, Loss: 244.5708
Epoch 27, Loss: 243.2532
Epoch 28, Loss: 241.9742
Epoch 29, Loss: 240.7303
Epoch 30, Loss: 239.5184
Epoch 31, Loss: 238.3358
Epoch 32, Loss: 237.1801
Epoch 33, Loss: 236.0491
Epoch 34, Loss: 234.9410
Epoch 35, Loss: 233.8541
Epoch 36, Loss: 232.7870
Epoch 37, Loss: 231.7383
Epoch 38, Loss: 230.7068
Epoch 39, Loss

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

def find_similar_words(word, word_to_idx, idx_to_word, word_embeddings, top_n=5):
    if word not in word_to_idx:
        print(f"'{word}' not in vocabulary.")
        return

    word_idx = word_to_idx[word]
    word_vec = word_embeddings[word_idx].reshape(1, -1)

    similarities = cosine_similarity(word_vec, word_embeddings)[0]

    # Sort by similarity and get top_n words (excluding the word itself)
    sorted_indices = np.argsort(similarities)[::-1]

    print(f"\nWords most similar to '{word}':")
    count = 0
    for i in sorted_indices:
        if idx_to_word[i] != word:
            print(f"  {idx_to_word[i]}: {similarities[i]:.4f}")
            count += 1
        if count >= top_n:
            break

# Example usage:
find_similar_words('books', word_to_idx, idx_to_word, word_embeddings)
find_similar_words('love', word_to_idx, idx_to_word, word_embeddings)
find_similar_words('morning', word_to_idx, idx_to_word, word_embeddings)



Words most similar to 'books':
  evening: 0.6179
  for: 0.3926
  are: 0.2912
  morning: 0.2153
  watch: 0.1976

Words most similar to 'love':
  i: 0.4431
  to: 0.3626
  in: 0.2190
  morning: 0.2130
  movies: 0.2004

Words most similar to 'morning':
  evening: 0.7693
  i: 0.3467
  watch: 0.3137
  and: 0.2494
  books: 0.2153
